# Binance USD-M Futures Partial Depth Service

This notebook shows how to call the local `BinanceFuturesDepthService` module from Jupyter. It subscribes to Binance USD-M Futures partial book depth streams, keeps the latest in-memory top-N depth snapshot per symbol, and reads bid1/bid2/ask1/ask2 through `get_latest_level_quote()`.

Run the cleanup cell at the end when you are done because the depth service starts a background receiver thread.

## 1. Import Local Package

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import BinanceDepthConfig, BinanceFuturesDepthService

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Configure Partial Depth Streams

`levels` must be `5`, `10`, or `20`. `speed_ms` must be `100`, `250`, or `500`; `250` maps to the no-suffix stream name, for example `btcusdt@depth5`.

In [2]:
config = BinanceDepthConfig(
    symbols=["BTCUSDT", "ETHUSDT"],
    levels=5,
    speed_ms=100,
    read_timeout_seconds=30.0,
    startup_timeout_seconds=30.0,
)

service = BinanceFuturesDepthService(config)
print(service.build_url())

wss://fstream.binance.com/public/stream?streams=btcusdt@depth5@100ms/ethusdt@depth5@100ms


## 3. Start The Background Receiver

This opens one real Binance WebSocket connection. `block_until_ready=True` waits until the WebSocket connection is established and there is no current connection-level error. A specific symbol can still be missing a snapshot, so check `service.get_latest(symbol)` before using it.

In [3]:
service.start(block_until_ready=True, timeout=30.0)
status = service.status()
print("ready:", status.ready)
print("connected:", status.connected)
print("symbols:", status.symbols)

ready: True
connected: True
symbols: ('BTCUSDT', 'ETHUSDT')


## 4. Read Bid1/Bid2/Ask1/Ask2

In [4]:
def quote_rows(symbol, max_level=2):
    rows = []
    sample_time_ms = time.time_ns() // 1_000_000

    for level in range(1, max_level + 1):
        quote = service.get_latest_level_quote(
            symbol,
            level=level,
            require_sequence_continuity=True,
        )
        if quote is None:
            continue

        quote_age_ms = sample_time_ms - quote.event_time_ms
        for side_name, price, qty in [
            ("bid", quote.bid_price, quote.bid_qty),
            ("ask", quote.ask_price, quote.ask_qty),
        ]:
            if price is None or qty is None:
                continue
            rows.append(
                {
                    "symbol": quote.symbol,
                    "side": side_name,
                    "level": quote.level,
                    "price": price,
                    "qty": qty,
                    "event_time_ms": quote.event_time_ms,
                    "local_recv_time_ms": quote.local_recv_time_ms,
                    "quote_age_ms": quote_age_ms,
                    "receive_latency_ms": quote.receive_latency_ms,
                    "is_stale": quote.is_stale,
                    "sequence_gap": quote.sequence_gap,
                    "final_update_id": quote.final_update_id,
                }
            )
    return rows


rows = []
for symbol in config.symbols:
    rows.extend(quote_rows(symbol, max_level=2))

display(pd.DataFrame(rows))

,symbol,side,level,price,qty,event_time_ms,local_recv_time_ms,quote_age_ms,receive_latency_ms,is_stale,sequence_gap,final_update_id
0,BTCUSDT,bid,1,73917.00,23.737,1780233350702,1780233350746,197,44,False,False,10671509447453
1,BTCUSDT,ask,1,73917.10,2.683,1780233350702,1780233350746,197,44,False,False,10671509447453
2,BTCUSDT,bid,2,73916.90,0.010,1780233350702,1780233350746,197,44,False,False,10671509447453
3,BTCUSDT,ask,2,73917.20,0.082,1780233350702,1780233350746,197,44,False,False,10671509447453
4,ETHUSDT,bid,1,2023.20,493.658,1780233350850,1780233350894,50,44,False,False,10671509472310
5,ETHUSDT,ask,1,2023.21,6.312,1780233350850,1780233350894,50,44,False,False,10671509472310
6,ETHUSDT,bid,2,2023.19,5.752,1780233350850,1780233350894,50,44,False,False,10671509472310
7,ETHUSDT,ask,2,2023.23,0.002,1780233350850,1780233350894,50,44,False,False,10671509472310


## Symbol-Indexed Depth DataFrame

`get_latest_depth_frame()` returns one row per subscribed symbol. Numeric bid/ask cells are `np.nan` when a symbol has no usable snapshot, is stale, fails the optional sequence check, or does not have that depth level.


In [ ]:
depth_frame = service.get_latest_depth_frame(
    levels=2,
    require_sequence_continuity=True,
)

display(depth_frame)


## 5. Inspect Full Top-N Snapshots

In [5]:
def snapshot_to_frame(snapshot):
    rows = []
    for side_name, levels in [("bid", snapshot.bids), ("ask", snapshot.asks)]:
        for level in levels:
            rows.append(
                {
                    "symbol": snapshot.symbol,
                    "side": side_name,
                    "level": level.level,
                    "price": level.price,
                    "qty": level.qty,
                    "spread": snapshot.spread,
                    "spread_bps": snapshot.spread_bps,
                    "mid_price": snapshot.mid_price,
                    "final_update_id": snapshot.final_update_id,
                }
            )
    return pd.DataFrame(rows)


for symbol in config.symbols:
    snapshot = service.get_latest(symbol)
    if snapshot is None:
        print(symbol, "has no snapshot yet")
        continue
    print(symbol, "stale=", snapshot.is_stale, "sequence_gap=", snapshot.sequence_gap)
    display(snapshot_to_frame(snapshot))

BTCUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,BTCUSDT,bid,1,73917.00,20.085,0.10,0.01352867843075447410306553089,73917.05,10671509703833
1,BTCUSDT,bid,2,73916.90,0.043,0.10,0.01352867843075447410306553089,73917.05,10671509703833
2,BTCUSDT,bid,3,73916.80,0.161,0.10,0.01352867843075447410306553089,73917.05,10671509703833
3,BTCUSDT,bid,4,73916.70,0.001,0.10,0.01352867843075447410306553089,73917.05,10671509703833
4,BTCUSDT,bid,5,73916.60,0.004,0.10,0.01352867843075447410306553089,73917.05,10671509703833
5,BTCUSDT,ask,1,73917.10,4.628,0.10,0.01352867843075447410306553089,73917.05,10671509703833
6,BTCUSDT,ask,2,73917.20,0.001,0.10,0.01352867843075447410306553089,73917.05,10671509703833
7,BTCUSDT,ask,3,73917.30,0.041,0.10,0.01352867843075447410306553089,73917.05,10671509703833
8,BTCUSDT,ask,4,73917.60,0.001,0.10,0.01352867843075447410306553089,73917.05,10671509703833
9,BTCUSDT,ask,5,73917.70,0.003,0.10,0.01352867843075447410306553089,73917.05,10671509703833


ETHUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,ETHUSDT,bid,1,2023.20,158.479,0.01,0.04942652870074955330774686698,2023.205,10671509699197
1,ETHUSDT,bid,2,2023.19,0.129,0.01,0.04942652870074955330774686698,2023.205,10671509699197
2,ETHUSDT,bid,3,2023.18,0.013,0.01,0.04942652870074955330774686698,2023.205,10671509699197
3,ETHUSDT,bid,4,2023.17,0.113,0.01,0.04942652870074955330774686698,2023.205,10671509699197
4,ETHUSDT,bid,5,2023.16,7.901,0.01,0.04942652870074955330774686698,2023.205,10671509699197
5,ETHUSDT,ask,1,2023.21,150.436,0.01,0.04942652870074955330774686698,2023.205,10671509699197
6,ETHUSDT,ask,2,2023.22,7.172,0.01,0.04942652870074955330774686698,2023.205,10671509699197
7,ETHUSDT,ask,3,2023.23,0.064,0.01,0.04942652870074955330774686698,2023.205,10671509699197
8,ETHUSDT,ask,4,2023.24,4.569,0.01,0.04942652870074955330774686698,2023.205,10671509699197
9,ETHUSDT,ask,5,2023.25,0.012,0.01,0.04942652870074955330774686698,2023.205,10671509699197


## 6. Watch Updates Briefly

This cell samples the in-memory latest table every second for 10 seconds. It does not write CSV, Parquet, or database records.

In [6]:
for _ in range(10):
    sample_time_ms = time.time_ns() // 1_000_000
    rows = []
    for symbol in config.symbols:
        quote = service.get_latest_level_quote(symbol, level=1)
        if quote is None:
            rows.append({"symbol": symbol, "state": "missing"})
            continue
        rows.append(
            {
                "symbol": symbol,
                "state": "stale" if quote.is_stale else "live",
                "sequence_gap": quote.sequence_gap,
                "bid1": quote.bid_price,
                "ask1": quote.ask_price,
                "quote_age_ms": sample_time_ms - quote.event_time_ms,
                "final_update_id": quote.final_update_id,
            }
        )
    display(pd.DataFrame(rows))
    time.sleep(1.0)

,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,77,10671511404365
1,ETHUSDT,live,False,2023.65,2023.66,71,10671511404865


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,45,10671511477992
1,ETHUSDT,live,False,2023.65,2023.66,57,10671511477278


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,135,10671511547429
1,ETHUSDT,live,False,2023.65,2023.66,135,10671511547396


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,55,10671511643615
1,ETHUSDT,live,False,2023.65,2023.66,103,10671511640814


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,186,10671511720288
1,ETHUSDT,live,False,2023.65,2023.66,60,10671511730051


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,148,10671511801197
1,ETHUSDT,live,False,2023.65,2023.66,130,10671511803955


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,96,10671511880934
1,ETHUSDT,live,False,2023.65,2023.66,114,10671511879580


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,106,10671511956028
1,ETHUSDT,live,False,2023.65,2023.66,118,10671511955057


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,49,10671512039074
1,ETHUSDT,live,False,2023.65,2023.66,103,10671512035684


,symbol,state,sequence_gap,bid1,ask1,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73936.30,73936.40,56,10671512104728
1,ETHUSDT,live,False,2023.65,2023.66,140,10671512098025


## 7. Stop The Background Thread

Always stop the service when you are done with the notebook kernel or before re-running the start cell.

In [7]:
service.stop(timeout=10.0)
service.status()

DepthServiceStatus(symbols=('BTCUSDT', 'ETHUSDT'), running=False, ready=False, connected=False, url='wss://fstream.binance.com/public/stream?streams=btcusdt@depth5@100ms/ethusdt@depth5@100ms', last_connect_at=datetime.datetime(2026, 5, 31, 7, 31, 57, 719637, tzinfo=datetime.timezone.utc), last_message_at=datetime.datetime(2026, 5, 31, 7, 32, 13, 476642, tzinfo=datetime.timezone.utc), reconnect_attempts=1, last_error=None)